[Reference](https://pub.towardsai.net/building-an-ai-powered-tweet-emotion-classifier-with-openai-a-complete-guide-8aa94d2c8174)

# pandas
scikit-learn
openai
matplotlib
seaborn

```
requirements.txt
pandas
scikit-learn
openai
matplotlib
seaborn
```

```
python -m venv .venv
source .venv/bin/activate
.venv\Scripts\activate
pip install -r requirements.txt
```

```
.env file
OPENAI_API_KEY=your-api-key-here
```

# Step 2: Import the Libraries

In [1]:
import os
import pandas as pd
from google import genai
from sklearn.metrics import accuracy_score
import random
import re
from openai import OpenAI
import matplotlib.pyplot as plt
import seaborn as sns

# Step 3: Load and Explore the Data

In [2]:
# Load the dataset
df = pd.read_csv('https://raw.githubusercontent.com/alphaiterations/agentic-ai-usecases/refs/heads/main/beginner/twitter-sentiment-classification/data/smile-annotations-final.csv', header=None, names=['tweet_id', 'tweet_text', 'emotion'])
print(f"Dataset shape: {df.shape}")
print(df.head())

Dataset shape: (3085, 3)
             tweet_id                                         tweet_text  \
0  611857364396965889  @aandraous @britishmuseum @AndrewsAntonio Merc...   
1  614484565059596288  Dorian Gray with Rainbow Scarf #LoveWins (from...   
2  614746522043973632  @SelectShowcase @Tate_StIves ... Replace with ...   
3  614877582664835073  @Sofabsports thank you for following me back. ...   
4  611932373039644672  @britishmuseum @TudorHistory What a beautiful ...   

  emotion  
0  nocode  
1   happy  
2   happy  
3   happy  
4   happy  


In [3]:
# checking the distribution of each emotion
df["emotion"].value_counts()

,count
emotion,
nocode,1572
happy,1137
not-relevant,214
angry,57
surprise,35
sad,32
happy|surprise,11
happy|sad,9
disgust|angry,7


# Step 4: Initialize the OpenAI Client

In [4]:
# Initialize the client
openai_api_key = os.getenv('OPENAI_API_KEY')

openai_client = OpenAI(api_key=openai_api_key)

# defining the OpenAI function
def call_openai(messages_list, response_format="text"):
    response = openai_client.chat.completions.create(
        model="gpt-5-nano",
        messages=messages_list,
        temperature=1,
        reasoning_effort="low",
        response_format={"type": response_format}
    )
    return response

# testing the function
messages_list = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Who won the World Series in 2020? Please answer in 20 words."}
]
response = call_openai(messages_list)

print(response)

In [5]:
print(response.choices[0].message.content)

# Step 5: Design the Emotion Classification Prompt


In [6]:
## Defining the Prompt
prompt = f"""Analyze the emotional content of the following tweet.

    For EACH of the following emotions, assign an intensity score between 0 and 1:
    - anger
    - disgust
    - happy
    - surprise
    - sad

    Where:
    0 = emotion not present
    1 = extremely strong presence of the emotion

    Return ONLY valid JSON in the exact format below:
    {{
    "anger": <number between 0 and 1>,
    "disgust": <number between 0 and 1>,
    "happy": <number between 0 and 1>,
    "surprise": <number between 0 and 1>,
    "sad": <number between 0 and 1>
    }}

    Do not include explanations, markdown, or extra text.

    Tweet: {{tweet_text}}"""

# Step 6: Build the Classification Function

In [7]:
import json

def classify_emotion(tweet_text, prompt):
    """
    Classify the emotional intensities of a tweet using Gemini API.
    Returns:
        emotions (dict): emotion -> intensity (0 to 1)
        input_tokens (int)
        output_tokens (int)
        raw_text (str): raw model output
    """
    prompt = prompt.replace("{tweet_text}", tweet_text)

    messages_list = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    response = call_openai(messages_list, response_format="json_object")
    #print(response)
    raw_text = response.choices[0].message.content

    # Token usage
    input_tokens = response.usage.prompt_tokens
    reasoning_tokens = response.usage.completion_tokens_details.reasoning_tokens
    output_tokens = response.usage.completion_tokens

    # Default fallback
    emotions = {
        "anger": 0.0,
        "disgust": 0.0,
        "happy": 0.0,
        "surprise": 0.0,
        "sad": 0.0,
    }


    try:
        # 🔑 Extract JSON object even if wrapped in text or markdown
        json_match = re.search(r"\{.*\}", raw_text, re.DOTALL)
        if not json_match:
            raise ValueError("No JSON object found in model output")

        parsed = json.loads(json_match.group())

        for key in emotions:
            if key in parsed:
                value = float(parsed[key])
                emotions[key] = max(0.0, min(1.0, value))

    except Exception as e:
        print("JSON parsing failed:", e)

    return emotions, input_tokens, reasoning_tokens, output_tokens,


# Test the function
test_tweet = "I love this beautiful painting!"
emotions, in_tok, reasoning_tokens, out_tok = classify_emotion(test_tweet, prompt)
print(f"Test: {emotions}, Input tokens: {in_tok}, Reasoning tokens: {reasoning_tokens}, Output tokens: {out_tok}")

# Step 7: Prepare Evaluation Dataset

In [8]:
# Explore emotion distribution
print(df['emotion'].value_counts())

# Filter to single emotions from the 5 classes
valid_emotions = ['angry', 'disgust', 'happy', 'surprise', 'sad']
df_filtered = df[df['emotion'].isin(valid_emotions)]
print(f"Filtered dataset shape: {df_filtered.shape}")
print(df_filtered['emotion'].value_counts())

In [9]:
# Sample 30 examples per emotion class for evaluation
sample_size = 30
sampled_df = df_filtered.groupby('emotion').apply(lambda x: x.sample(min(len(x), sample_size))).reset_index(drop=True)
print(f"Sampled dataset shape: {sampled_df.shape}")
print(sampled_df['emotion'].value_counts())

# Step 8: Run Classification on Test Set


In [10]:
# Classify the sampled tweets
predictions = []
input_tokens_list = []
reasoning_tokens_list = []
output_tokens_list = []
raw_predictions_list = []
predicted_emotion_score_list = []

# Mapping from model emotions to dataset labels
emotion_mapping = {
    'anger': 'angry',
    'disgust': 'disgust',
    'happy': 'happy',
    'surprise': 'surprise',
    'sad': 'sad'
}

for idx, row in sampled_df.iterrows():
    tweet = row['tweet_text']
    true_emotion = row['emotion']

    emotions, in_tok, reasoning_tokens, out_tok = classify_emotion(tweet, prompt)
    # Select the emotion with the highest intensity
    predicted_key = max(emotions, key=emotions.get)
    predicted_emotion = emotion_mapping.get(predicted_key, predicted_key)
    predicted_score = emotions.get(predicted_key, 0.0)

    predictions.append(predicted_emotion)
    input_tokens_list.append(in_tok)
    reasoning_tokens_list.append(reasoning_tokens)
    output_tokens_list.append(out_tok)
    raw_predictions_list.append(emotions)

    predicted_emotion_score_list.append(predicted_score)

    print(f"Tweet {idx+1}: True: {true_emotion}, Predicted: {predicted_emotion} (scores: {emotions}), Tokens: {in_tok}+{out_tok}")

sampled_df['predicted_emotion'] = predictions
sampled_df['input_tokens'] = input_tokens_list
sampled_df['output_tokens'] = output_tokens_list
sampled_df['raw_prediction'] = raw_predictions_list
sampled_df['predicted_emotion_score'] = predicted_emotion_score_list

total_input_tokens = sum(input_tokens_list)
total_output_tokens = sum(output_tokens_list)
print(f"Total input tokens: {total_input_tokens}, Total output tokens: {total_output_tokens}")

In [11]:
sampled_df.head()

# Step 9: Evaluate Model Performance


In [12]:
# Calculate accuracy
# Since the model outputs intensity scores for all emotions, we evaluate by selecting the emotion with the highest score
accuracy = accuracy_score(sampled_df['emotion'], sampled_df['predicted_emotion'])
print(f"Accuracy on sampled data: {accuracy:.2%}")

# Per-class accuracy
for emotion in valid_emotions:
    subset = sampled_df[sampled_df['emotion'] == emotion]
    if len(subset) > 0:
        acc = accuracy_score(subset['emotion'], subset['predicted_emotion'])
        print(f"Accuracy for {emotion}: {acc:.2%} ({len(subset)} samples)")

In [13]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 5)

# 1. Overall Accuracy
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Overall Accuracy Bar Chart
overall_acc = accuracy_score(sampled_df['emotion'], sampled_df['predicted_emotion'])
axes[0].bar(['Overall Accuracy'], [overall_acc], color='steelblue', alpha=0.7)
axes[0].set_ylim([0, 1])
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Overall Model Accuracy', fontsize=13, fontweight='bold')
axes[0].text(0, overall_acc + 0.02, f'{overall_acc:.2%}', ha='center', fontsize=11, fontweight='bold')

# Plot 2: Per-Class Accuracy
emotion_accuracies = []
emotion_labels = []
for emotion in valid_emotions:
    subset = sampled_df[sampled_df['emotion'] == emotion]
    if len(subset) > 0:
        acc = accuracy_score(subset['emotion'], subset['predicted_emotion'])
        emotion_accuracies.append(acc)
        emotion_labels.append(emotion)

colors = plt.cm.Set3(range(len(emotion_labels)))
bars = axes[1].bar(emotion_labels, emotion_accuracies, color=colors, alpha=0.7)
axes[1].set_ylim([0, 1])
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Per-Class Accuracy', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
for bar, acc in zip(bars, emotion_accuracies):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{acc:.0%}', ha='center', va='bottom', fontsize=9)

# Plot 3: Sample Distribution
sample_counts = sampled_df['emotion'].value_counts().reindex(emotion_labels, fill_value=0)
axes[2].bar(emotion_labels, sample_counts, color=colors, alpha=0.7)
axes[2].set_ylabel('Number of Samples', fontsize=12)
axes[2].set_title('Sample Distribution by Emotion', fontsize=13, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45)
for i, (label, count) in enumerate(zip(emotion_labels, sample_counts)):
    axes[2].text(i, count + 0.5, str(count), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n✓ Overall Accuracy: {overall_acc:.2%}")
for emotion, acc in zip(emotion_labels, emotion_accuracies):
    subset = sampled_df[sampled_df['emotion'] == emotion]
    print(f"✓ {emotion}: {acc:.2%} ({len(subset)} samples)")

# Step 10: Analyze API Costs

In [14]:
# Cost estimation
# gpt-5-nano pricing (as of 2026): $0.05 per 1M input tokens, $0.40 per 1M output tokens
input_cost_per_million = 0.05
output_cost_per_million = 0.40

input_cost = (total_input_tokens / 1_000_000) * input_cost_per_million
output_cost = (total_output_tokens / 1_000_000) * output_cost_per_million
total_cost_sample = input_cost + output_cost

print(f"Cost for {len(sampled_df)} samples: ${total_cost_sample:.4f}")

# Extrapolate to full dataset
full_dataset_size = len(df_filtered)
scale_factor = full_dataset_size / len(sampled_df)
estimated_input_tokens = total_input_tokens * scale_factor
estimated_output_tokens = total_output_tokens * scale_factor
estimated_cost = (estimated_input_tokens / 1_000_000) * input_cost_per_million + (estimated_output_tokens / 1_000_000) * output_cost_per_million

print(f"Estimated cost for full dataset ({full_dataset_size} tweets): ${estimated_cost:.2f}")
print(f"Estimated tokens: Input {estimated_input_tokens:.0f}, Output {estimated_output_tokens:.0f}")